# Knowledge Graph QA Pipeline

This notebook is the recruiter-facing walkthrough for a natural-language question answering pipeline over an RDF knowledge graph. It demonstrates how a user question is parsed into graph structure, linked to entities, expanded into a local subgraph, converted to SPARQL, and returned as readable answers.

The reusable implementation lives in `src/kg_query_pipeline`; this notebook keeps the narrative and demo workflow lightweight.

## Project Snapshot

**Problem:** answer natural-language questions from a Turtle/RDF knowledge graph.

**Approach:** combine LLM query parsing, embedding-based entity/predicate matching, local subgraph extraction, SPARQL execution, and answer materialization.

**Core stack:** OpenAI API, RDFLib, SentenceTransformers, NetworkX, Pandas.

**Current evaluation note:** the original run completed 12/12 curated questions without runtime errors. Answer accuracy should be reported only after manual ground-truth review.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from dotenv import load_dotenv
import pandas as pd

from kg_query_pipeline import KGQAPipeline, PipelineConfig

load_dotenv(ROOT / ".env")

GRAPH_PATH = ROOT / "data" / "output_graph.ttl"
QUESTIONS_PATH = ROOT / "data" / "questions_set_easy.json"
RESULTS_PATH = ROOT / "results" / "batch_results.csv"

print(f"Graph path: {GRAPH_PATH}")
print(f"Graph available: {GRAPH_PATH.exists()}")

## Pipeline Architecture

```text
Natural-language question
        -> LLM query parser
        -> entity linker
        -> local subgraph extraction
        -> predicate matcher
        -> SPARQL generation
        -> RDFLib execution
        -> readable answer
```

The package path is intentionally local-first. No Google Drive mount or Colab-specific path is required.

## Curated Question Set

The curated questions are stored as JSON so the same asset can be used by the notebook and by `scripts/run_batch.py`.

In [ ]:
questions_df = pd.read_json(QUESTIONS_PATH)
questions_df

## Single-Question Demo

Set `OPENAI_API_KEY` in `.env` before running this cell. The graph should be available at `data/output_graph.ttl`.

In [ ]:
question = "Who directed The Family Man?"

if GRAPH_PATH.exists():
    pipeline = KGQAPipeline(PipelineConfig(graph_path=GRAPH_PATH))
    result = pipeline.run(question)
    result
else:
    print("Add the Turtle graph at data/output_graph.ttl before running the end-to-end demo.")

## Batch Demo

The batch runner writes a recruiter-friendly CSV with the question, parsed structure, matched entity, generated answers, and any runtime error.

In [ ]:
if GRAPH_PATH.exists():
    pipeline = KGQAPipeline(PipelineConfig(graph_path=GRAPH_PATH))
    rows = []
    for question in questions_df["Question"]:
        result = pipeline.run(question)
        rows.append({
            "question": result.question,
            "answers": "; ".join(result.answers),
            "matched_entity": result.matched_entity,
            "error": result.error,
        })

    batch_df = pd.DataFrame(rows)
    batch_df.to_csv(RESULTS_PATH, index=False)
    batch_df
else:
    print("Batch demo skipped because data/output_graph.ttl is not present.")

## How To Interpret Results

A successful run means the pipeline completed without an exception. For a portfolio claim, add a manual review column to `results/batch_results.csv` and report answer accuracy separately from runtime success.

The sample result shape is documented in `results/batch_results_sample.csv`.

## Next Improvements

- Add ground-truth answers and a scoring script for exact match / F1-style evaluation.
- Normalize graph URI construction for entities with punctuation or Unicode characters.
- Support additional RDF namespaces through config instead of fixed defaults.
- Add tests for query parsing, entity matching, SPARQL generation, and fallback behavior.